In [2]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

在计算y关于**x**的梯度之前，需要一个地方来存储梯度

In [3]:
x.requires_grad_(True) #等价于 x = torch.arange(4.0, requires_grad=True)
x.grad #默认值为None

现在计算y

In [4]:
y = 2 * torch.dot(x,x)
y

tensor(28., grad_fn=<MulBackward0>)

通过调用反向传播函数自动计算y关于x每个分量的梯度

In [5]:
y.backward() #自动求导
x.grad #查看导数

tensor([ 0.,  4.,  8., 12.])

默认情况下，pytorch会累积梯度，因此需要清除之前的值

In [ ]:
x.grad.zero_() #梯度清零
y = x.sum()
y.backward()
x.grad

tensor([0., 0., 0., 0.])

当y不是标量时，向量y关于向量x的导数是一个雅可比矩阵。在深度学习里，我们其实并不关心这个大矩阵，我们最终关心的只有一个东西：**损失函数**。

方法1：这是最常用的办法。把 $y$ 里的所有元素加起来变成一个总和（标量）。加法的导数是 1，所以这样做相当于告诉程序：“我平等地看待 $y$ 里的每一个元素，请帮我算出它们累加后的总梯度。”

In [17]:
x.grad.zero_()
y = x * x
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

方法2：传入一个“权重说明书”（即 gradient 参数），该参数指定微分函数关于self的梯度。

这行代码的意思是：“虽然 $y$ 是个向量，但请你假设后面还有一个 Loss，$y$ 里的每个元素对 Loss 的影响权重都是 1。” 本例只想求偏导数的和，所以传递一个1的梯度是合适的

In [20]:
x.grad.zero_()
y = x * x
y.backward(torch.ones(len(x))) # 等价于y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

将某些计算移动到记录的计算图以外

In [ ]:
x.grad.zero_()
y = x * x
u = y.detach() #u的数值与y相等，但它认为自己与x无关
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [22]:
x.grad.zero_() 
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

即使构建函数的计算图需要通过Python控制流（例如，条件、循环或任意函数调用），我们仍然可以计算得到的变量的梯度。 在下面的代码中，while循环的迭代次数和if语句的结果都取决于输入a的值。

In [33]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()
a.grad == d / a

tensor(True)